In [ ]:
# IPython magig  tools
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import json
from scipy.stats import ttest_rel, ttest_ind
from aind_vr_foraging_analysis.utils.parsing import data_access, parse
import aind_vr_foraging_analysis.utils.plotting as plotting
import aind_vr_foraging_analysis.utils as processing
import aind_vr_foraging_analysis.utils.plotting.lick_functions as lick_functions

# Plotting libraries
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages

import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

sns.set_context('talk')
from pathlib import Path
import warnings
pd.options.mode.chained_assignment = None  # Ignore SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Rectangle

color1='#d95f02'
color2='#1b9e77'
color3='#7570b3'
color4='yellow'
odor_list_color = [color1, color2, color3, color4]

pdf_path = r'Z:\scratch\vr-foraging\sessions'
foraging_figures = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents'

from scipy.optimize import curve_fit

color_dict_label = {'InterSite': '#808080',
    'InterPatch': '#b3b3b3', 
    'PatchZ': '#d95f02', 'PatchZB': '#d95f02', 
    'PatchZA': '#d95f02', 
    'PatchB': '#d95f02','PatchA': '#7570b3', 
    'PatchC': '#1b9e77',
    'Alpha-pinene': '#1b9e77', 
    'Methyl Butyrate': '#d95f02', 
    'Amyl Acetate':  '#7570b3',
    'Fenchone': '#7570b3', 
    'S': color1,
    'D': color2,
    'N': color3,
    "odor_90": color1,
    "odor_60": color2,
    "odor_0": color3,
    'A': color1,
    'B': color2,
    'C': color3,
    'OdorA': color1,
    'OdorB': color2,
    'OdorC': color3,
    'odor_slow': color1,
    'odor_fast': color2,
    'A': color1,
    'B1': color2,
    'B2': color2,
    'C1': color3,
    'C2': color4
    }

label_dict = {**{
"InterSite": '#808080',
"InterPatch": '#b3b3b3'}, 
            **color_dict_label}

from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d


In [ ]:
import contraqctor
import logging
from dataclasses import dataclass, field

@dataclass
class ProcessedLickometer:
    onsets: np.ndarray
    frequency: pd.DataFrame
    
def process_lickometer(
    data, *, refractory_period_s: float = 0.05, dt_resample: float = 0.1
) -> ProcessedLickometer | None:
    from contraqctor.qc.harp import lickety_split

    data['harp_lickometer'].streams.LickState.load_from_file()
    data['harp_lickometer'].streams.TimestampSeconds.load_from_file()
    lickometer = data['harp_lickometer'].streams.LickState.data.copy()
    lickometer = lickometer[lickometer["MessageType"] == "EVENT"]["Channel0"]
    lick_onsets = lickometer[(lickometer) & (~lickometer.shift(1, fill_value=False))].index
    if len(lick_onsets) == 0:
        logging.warning("No lick onsets found in lickometer data.")
        return None

    suite = lickety_split.HarpLicketySplitTestSuite(
        data['harp_lickometer'].streams, lick_refractory_period=refractory_period_s
    )
    test_minimum_lick_rate = suite.test_minimum_lick_rate()
    if test_minimum_lick_rate.status != contraqctor.qc.Status.PASSED:
        logging.warning(
            f"Lickometer data quality test failed: Minimum Lick Rate Test - {test_minimum_lick_rate.result}"
        )
        return None

    keep = np.ones(len(lick_onsets), dtype=bool)
    keep[1:] = np.diff(lick_onsets) >= refractory_period_s
    kept = lick_onsets[keep]

    t_start = lickometer.index.values[0]
    t_end = lickometer.index.values[-1]

    bin_edges = np.arange(t_start, t_end + dt_resample, dt_resample)
    counts, _ = np.histogram(kept, bins=bin_edges)

    frequency_hz = counts / dt_resample

    bin_centers = bin_edges[:-1] + dt_resample / 2

    frequency = pd.Series(
        frequency_hz,
        index=pd.Index(bin_centers, name="Seconds"),
        name="frequency",
    )

    return ProcessedLickometer(
        onsets=kept,
        frequency=frequency,
    )

In [ ]:
def lick_rate_calculation(stream_data, dt_resample=250, sigma_s=0.25, t_start=None, t_end=None):
    """
    Compute smoothed lick rate on a uniform time base.

    Args:
        stream_data:   object with a .lick_onset attribute (indexed by Time)
        dt_resample:   sampling rate of the output (Hz)
        sigma_s:       smoothing kernel width in seconds (default 250ms)
        t_start:       start of time base (defaults to first lick time)
        t_end:         end of time base (defaults to last lick time)
    """
    licks = stream_data.lick_onset.reset_index()
    lick_times = licks['Time'].values

    if len(lick_times) < 2:
        raise ValueError("Need at least 2 licks to compute lick rate.")

    # Build uniform time base, defaulting to lick range if not provided
    t_start = t_start if t_start is not None else lick_times[0]
    t_end   = t_end   if t_end   is not None else lick_times[-1]

    dt = 1 / dt_resample
    t_uniform = np.arange(t_start, t_end + dt, dt)

    lick_signal = np.zeros_like(t_uniform)

    # Place instantaneous rate at the midpoint of each inter-lick interval
    for i in range(len(lick_times) - 1):
        ili       = lick_times[i + 1] - lick_times[i]   # inter-lick interval (s)
        t_mid     = (lick_times[i] + lick_times[i + 1]) / 2
        idx       = np.searchsorted(t_uniform, t_mid)
        idx       = min(idx, len(t_uniform) - 1)
        lick_signal[idx] += 1 / ili                      # instantaneous rate (Hz)

    # Smooth with Gaussian kernel (sigma in samples)
    sigma_samples = sigma_s * dt_resample
    lick_rate_smoothed = gaussian_filter1d(lick_signal, sigma=sigma_samples)

    return pd.DataFrame({
        'lick_rate': lick_rate_smoothed
    }, index=pd.Index(t_uniform, name='Time'))

In [ ]:
import math

def make_plot(df, save = None, aligned_to = 'choice_cue_time'):
    plot_configs = [
        {
            'data': df,
            'hue': 'is_reward',
            'palette': ['crimson', 'darkgreen'],
            'legend_title': 'Reward',
            'title': 'All trials — reward'
        },
        {
            'data': df.loc[
                (df.is_reward == 1) &
                (df.reward_probability >= 0.2) &
                (df.patch_label == "odor_90") &
                (df.total_sites>=3)
            ],
            'hue': 'reward_probability',
            'palette': 'magma_r',
            'legend_title': 'Reward probability',
            'title': 'Reward — Odor 90 by site'
        },
        {
            'data': df.loc[
                (df.is_reward == 1) &
                (df.reward_probability >= 0.2) &
                (df.patch_label == "odor_60") & 
                (df.total_sites>=3)
            ],
            'hue': 'reward_probability',
            'palette': 'magma_r',
            'legend_title': 'Reward probability',
            'title': 'Reward — Alpha-pinene by site'
        },
        {
            'data': df.loc[df.is_reward == 0],
            'hue': 'patch_label',
            'palette': color_dict_label,
            'legend_title': 'Patch label',
            'title': 'No-reward trials by patch'
        },
        {
            'data': df.loc[(df.is_reward == 0)&(df.trials_to_leave>-10)&(df.total_sites>=3)],
            'hue': 'trials_to_leave',
            'palette': 'magma_r',
            'legend_title': 'Trials before leaving',
            'title': 'No-reward trials by trials to leave'
        },
        {
            'data': df.loc[df.is_reward == 1],
            'hue': 'patch_label',
            'palette': color_dict_label,
            'legend_title': 'Patch label',
            'title': 'Reward trials by patch'
        },
        {
            'data': df.loc[(df.is_reward == 1)&(df.trials_to_leave>-10)&(df.total_sites>=3)],
            'hue': 'trials_to_leave',
            'palette': 'magma_r',
            'legend_title': 'Trials before leaving',
            'title': 'Reward trials by trials to leave',
            'window': (0, 2)
        },
        {
            'data': df.loc[(df.site_number == 0)],
            'hue': 'patch_label',
            'palette': color_dict_label,
            'legend_title': 'Patch label',
            'title': 'Trials (site 0 only)',
            'window': (0, 2)
        },
        {
            'data': df.loc[
                (df.patch_label == "odor_90")  &
                (df.reward_probability >= 0.2) &
                (df.total_sites>=3)
            ],
            'hue': 'reward_probability',
            'palette': 'magma_r',
            'legend_title': 'Reward probability',
            'title': 'Odor 90 by reward probability',
            'window': (0, 2)
        },
        {
            'data': df.loc[
                (df.patch_label == "odor_60")  &
                (df.reward_probability >= 0.2) &
                (df.total_sites>=3)
            ],
            'hue': 'reward_probability',
            'palette': 'magma_r',
            'legend_title': 'Reward probability',
            'title': 'Odor 60 by reward probability',
            'window': (0, 2)
        },
        {
            'data': df.loc[(df.is_reward == 1)&(df.patch_label == 'odor_90')],
            'hue': 'previous_outcome',
            'palette': ['indigo', 'orange'],
            'legend_title': 'Previous outcome',
            'title': 'Reward trials by previous outcome'
        },
        {
            'data': df.loc[(df.is_reward == 1)&(df.patch_label == 'odor_60')],
            'hue': 'previous_outcome',
            'palette': ['indigo', 'orange'],
            'legend_title': 'Previous outcome',
            'title': 'Reward trials by previous outcome'
        },
        ]

    # --- Grid layout ---
    n_cols = 3
    n_rows = math.ceil(len(plot_configs) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    ymax = df['lick_rate'].max()

    for i, (cfg, ax) in enumerate(zip(plot_configs, axes)):
        sns.lineplot(
            data=cfg['data'],
            x='times',
            y='lick_rate',
            hue=cfg['hue'],
            errorbar='sd',
            palette=cfg['palette'],
            err_kws={'linewidth': 0},
            ax=ax
        )
        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        if 'window' in cfg:
            ax.set_xlim(cfg['window'])
        else:
            ax.set_xlim(-2, 5)
        # ax.set_ylim(-0.5, ymax)
        ax.set_ylim(-0.5, 10)
        
        if aligned_to == 'choice_cue_time':
            ax.set_xlabel('Time from choice cue (s)')
        elif aligned_to == 'odor_onset':
            ax.set_xlabel('Time from odor onset (s)')
        else:
            ax.set_xlabel('Time from reward (s)')
            
        ax.set_ylabel('Lick Rate (licks/s)')
        ax.set_title(cfg['title'], fontsize=10)
        ax.legend(
            bbox_to_anchor=(1.05, 1), loc='upper left',
            title=cfg['legend_title'], fontsize=8, title_fontsize=8
        )
        sns.despine(ax=ax)

    if df.session.nunique() == 1:
        plt.suptitle(df.session.unique()[0], fontsize=16)
    else:
        plt.suptitle(f"{df.session.nunique()} sessions", fontsize=16)
    # Hide any unused axes
    for j in range(len(plot_configs), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    if save:
        save.savefig(fig)
        plt.close()
    else:
        plt.show()

### Check individual sessions

In [ ]:
from aind_vr_foraging_analysis.utils.parsing import data_access
date_string = "2024-12-16" # YYYY-MM-DD
mouse = '745302' # mouse ID
df = pd.DataFrame()

# This section will look at all the session paths that fulfill the condition
session_paths = data_access.find_sessions_relative_to_date(
    mouse=mouse,
    date_string=date_string,
    when='on'
)
# Iterate over the session paths and load the data
for session_path in session_paths:
    # print(f"Loading {session_path.name}...")
    # try:
    all_epochs, stream_data, data = data_access.load_session(
        session_path, extra=True
    )
    print(f"Loaded {session_path.name} successfully.")
    print(data['software_events'].streams.GiveReward.data['data'].sum())
    print(data['config'].streams.rig_input.data['rig_name'])
    
    odor_sites = all_epochs.loc[all_epochs['label'] == 'OdorSite']
    odor_sites['trials_to_leave'] = (
            odor_sites[odor_sites['is_choice'] == 1]
            .groupby('patch_number')['is_choice']
            .transform(lambda x: x.astype(int).cumsum() - x.astype(int).sum() - 1)
        )
    odor_sites['session'] = session_path.name
    df = pd.concat([df, odor_sites])
    
    total_rewards = odor_sites.is_reward.sum()
    total_licks = len(stream_data.lick_onset)
    metric = total_licks / total_rewards
    if metric < 20:
        print(f"Warning: Not enough licks per reward in session {session_path.name} (licks/reward = {metric:.2f}). Consider excluding this session from analysis.")
        continue 
    
    odor_sites.sort_index(inplace=True)
    max_prob = odor_sites.groupby(['session', 'odor_label'])['reward_probability'].transform('max')

    odor_sites['patch_label'] = max_prob.apply(lambda x: 'odor_90' if x > 0.75 else ('odor_60' if x > 0.3 else 'odor_0'))

    odor_sites['previous_outcome'] = odor_sites['is_reward'].shift(1)
    odor_sites.loc[odor_sites['site_number'] == 0, 'previous_outcome'] = np.nan
    odor_sites = odor_sites[odor_sites['is_choice'] == 1]
        
    lick_rate = lick_rate_calculation(stream_data, 10)
    
    odor_sites['total_sites'] = odor_sites.groupby('patch_number')['site_number'].transform('max') + 1
    
    trial_summary = plotting.trial_collection(odor_sites, lick_rate, cropped_to_length='epoch', window= [-4,4],
                                                aligned = 'choice_cue_time', taken_col='lick_rate')
    
    make_plot(trial_summary)
    

In [ ]:
expected_mean_rate = len(stream_data.lick_onset) / (stream_data.lick_onset.index[-1] - stream_data.lick_onset.index[0])
computed_mean_rate = lick_rate['lick_rate'].mean()
print(f"Expected: {expected_mean_rate:.2f} Hz, Computed: {computed_mean_rate:.2f} Hz")

In [ ]:
lick_times = stream_data.lick_onset.index.values

fig, ax = plt.subplots(2,1, figsize=(12, 3), sharex=True)
win = 1600
ax[0].vlines(lick_times, 0, 1, color='gray', alpha=0.4, label='lick onsets')
# ax[0].plot(t_uniform, lick_signal, color='steelblue', alpha=0.6, label='raw ILI rate')
ax[1].plot(lick_rate.index, lick_rate['lick_rate'], color='tomato', lw=2, label='smoothed')
ax[0].set(xlabel='Time (s)', ylabel='Lick rate (Hz)', title='Lick rate validation')
ax[0].legend()
ax[0].set_xlim(lick_times[0]+ win, lick_times[0] + win + 1)

In [ ]:
sns.regplot(x='reward_probability', y='trials_to_leave', data=trial_summary.loc[(trial_summary['patch_label'] == "Methyl Butyrate") & (trial_summary['total_sites'] >= 4)])

In [ ]:
sns.lineplot(data=trial_summary.loc[(trial_summary['patch_label'] == "Methyl Butyrate") & (trial_summary['trials_to_leave'] == -1)& (trial_summary['total_sites'] >= 1)], 
             x='times', 
             y='lick_rate', 
             palette='magma_r',
             hue='total_sites', marker = '')
sns.despine()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(-2, 5)

In [ ]:
sns.lineplot(data=trial_summary.loc[(trial_summary['patch_label'] == "Methyl Butyrate") & (trial_summary['is_reward'] == 1)& (trial_summary['total_sites'] >= 4)], 
             x='times', 
             y='lick_rate', 
             palette='magma_r', err_kws={'linewidth': 0},
             hue='reward_probability', marker = '')
sns.despine()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(-2, 5)

In [ ]:
sns.lineplot(data=trial_summary.loc[(trial_summary['patch_label'] == "Methyl Butyrate") & (trial_summary['reward_probability'] >= 0.3)& (trial_summary['reward_probability'] <= 0.4)& (trial_summary['trials_to_leave'] >= -4)], 
             x='times', 
             y='lick_rate', 
             palette='magma_r',
             hue='trials_to_leave', marker = '')
sns.despine()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(-2, 5)

In [ ]:
df_plot = all_mice_df.loc[
    (all_mice_df['is_reward'] == 1) 
    &  (all_mice_df['trials_to_leave'] >= -7)
].copy()

df_plot['reward_probability'] = df_plot['reward_probability'].round(2)

sns.histplot(
    df_plot,
    x='reward_probability',
    hue='trials_to_leave',
    multiple='stack',
    palette='Set1',
    bins=np.linspace(0, 1, 20),  # 20 evenly spaced bins from 0 to 1
    shrink=0.9
)
sns.despine()

### Explore the session 

In [ ]:
def add_position(df: pd.DataFrame, position:  pd.DataFrame):
    position.rename_axis('Time', axis='index', inplace=True)
    df.rename_axis('Time', axis='index', inplace=True)

    df = pd.merge_asof(df.sort_index(), position.sort_index(), direction='nearest', on="Time").set_index("Time").sort_index()
    df.columns = [*df.columns[:-1], 'Position']
    return df

def plot_patches(ax, navigation_mode, zero_index, _legend):
    """Plot the context patches based on the navigation mode."""
    _sites = add_position(all_epochs, position=stream_data.position_data)
    for idx, site in enumerate(all_epochs.iloc[:-1].iterrows()):
        site_label = site[1]["label"]
        if site_label == "Reward":
            site_label = f"Odor {site[1]['odor']['index'] + 1}"
            facecolor = label_dict[site_label]
        elif site_label == "OdorSite":
            site_label = site[1]['patch_label']
            facecolor = label_dict[site_label]
        elif site_label == "InterPatch":
            facecolor = label_dict[site_label]
        else:
            site_label = "InterSite"
            facecolor = label_dict["InterSite"]

        if navigation_mode == "space":
            position = _sites["Position"].values[idx]
            width = _sites["Position"].values[idx + 1] - position
            p = Rectangle((position, -2), width, 8, linewidth=0, facecolor=facecolor, alpha=0.5)
        else:
            time_position = all_epochs.index[idx] - zero_index
            width = all_epochs.index[idx + 1] - all_epochs.index[idx]
            p = Rectangle((time_position, -2), width, 8, linewidth=0, facecolor=facecolor, alpha=0.5)

        _legend[site_label] = p
        ax.add_patch(p)

def plot_behavioral_events(ax, navigation_mode,  zero_index, _legend):
    """
    Plot the behavioral events based on the selected navigation mode.
    """
    events = [
        ("SoftReward", data['software_events'].streams['GiveReward'].data, 3.5, 'red', 'x', 400),
        ("SoftTone", data['software_events'].streams['ChoiceFeedback'].data, 2.5, 'yellow', 'x', 400),
        ('ChoiceFeedback', stream_data.choice_feedback, 2.5, 'k', 's', 100),
        ('Lick', stream_data.lick_onset, 1, 'k', '|', 100),
        ('Reward', stream_data.give_reward, 3.5, 'mediumblue', '*',100),       
    ]
    
    # _legend["Waits"] = axs.scatter(stream_data.succesfull_wait.index - zero_index,
    #     succesfull_wait.index*0 + 1.2,
    #     marker=".", s=s, lw=lw, c='green',
    #     label="Reward")
    
    for event_name, event_data, y_pos, color, marker, size in events:
        if navigation_mode == "time" or navigation_mode == "patch":
            # Plot events by time index
            _legend[event_name] = ax.scatter(event_data.index-zero_index, [y_pos] * len(event_data.index), label=event_name, color=color, marker=marker, s=size)
        else:
            # Plot events by position (space or patch mode)
            positions = add_position(event_data, stream_data.position_data)['Position'].values
            _legend[event_name] = ax.scatter(positions, [y_pos] * len(positions), label=event_name, color=color, marker=marker, s=size)

In [ ]:
lick_rate = stream_data.lick_rate.copy()

In [ ]:
# Define a dictionary to map navigation modes to corresponding limits and increments
nav_config = {
    "patch": {"increment": 1},  # For patches, we move one patch at a time
    "time": {"increment": 20},
    "space": {"increment": 750}
}

def update_plot(x_start, navigation_mode="time"):
    zero_index = all_epochs.index[0]

    # Create figure with two axes stacked vertically
    fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(20, 6), sharex=True,
                                            gridspec_kw={'height_ratios': [1, 2]})
    
    _legend = {}

    # --- TOP AXIS: Lick rate ---
    # Assuming lick_rate_250Hz is a DataFrame with 'Time' and 'lick_rate'
    if navigation_mode == "time" or navigation_mode == "patch":
        ax_top.plot(
            lick_rate.index - zero_index,
            lick_rate['lick_rate'],
            c='b',
            label='Lick rate (Hz)'
        )
    else:  # navigation_mode == "space"
        # You need a spatial interpolation if you want x-axis in space
        ax_top.plot(
            add_position(lick_rate['lick_rate'], position=stream_data.position_data)["Position"].values,
            lick_rate['lick_rate'].values,
            c='b',
            label='Lick rate (Hz)'
        )
    ax_top.set_ylabel("Lick rate (Hz)")
    ax_top.legend()
    ax_top.grid(False)
    ax_top.set_ylim(-1,20)
    ax_top_twin = ax_top.twinx()
    plot_patches(ax_top_twin, navigation_mode, zero_index, _legend)

    # --- BOTTOM AXIS: Existing behavioral + velocity plot ---
    ax = ax_bottom  # for simplicity, keep your old code mostly unchanged
    nav_mode = nav_config.get(navigation_mode, nav_config["time"])

    if navigation_mode == "patch":
        patch_start_times = all_epochs.loc[all_epochs['label'] == 'InterPatch'].index
        patch_end_times = patch_start_times[1:].append(pd.Index([all_epochs.index[-1]]))
        patches = list(zip(patch_start_times, patch_end_times))
        current_patch = int(x_start)
        start, end = patches[current_patch]
        ax.set_xlim(start - zero_index, end - zero_index)
    else:
        ax.set_xlim(x_start, x_start + nav_mode.get("increment"))

    # Plot patches, behavioral events, velocity
    plot_patches(ax, navigation_mode, zero_index, _legend)
    plot_behavioral_events(ax, navigation_mode, zero_index, _legend)

    ax2 = ax.twinx()
    if navigation_mode in ["time", "patch"]:
        _legend["Velocity"] = ax2.plot(
            stream_data.encoder_data.index - zero_index,
            stream_data.encoder_data.filtered_velocity,
            c="k", label="Velocity", alpha=0.8
        )[0]
    else:
        _legend["Velocity"] = ax2.plot(
            add_position(stream_data.encoder_data.filtered_velocity, position=stream_data.position_data)["Position"].values,
            stream_data.encoder_data.filtered_velocity.values,
            c="k", label="Encoder", alpha=0.8
        )[0]

    # Bottom axis styling
    ax.set_yticklabels([])
    ax.set_yticks([])
    ax.set_xlabel("Time (s)" if navigation_mode != "space" else "VR Space (cm)")
    ax2.set_ylabel("Velocity (cm/s)")
    ax.set_ylim(bottom=-1, top=4)
    ax.set_yticks([0, 3])
    ax2.axhline(5, color='gray', linestyle='--', alpha=0.5)
    ax2.axhline(8, color='red', linestyle='--', alpha=0.5)
    ax2.yaxis.tick_left()
    ax2.yaxis.set_label_position("left")
    ax.grid(False)
    ax2.set_ylim((-10, 70))
    ax.legend(_legend.values(), _legend.keys(), bbox_to_anchor=(1.05, 0.5),
              loc='center left', borderaxespad=0.)

    plt.tight_layout()
    return fig, ax_bottom  # return figure if needed for saving
# Define callback functions for the arrow buttons with different navigation logic based on the plot type
def on_left_button_clicked(button):
    # Adjust the x_start based on the selected navigation mode
    if navigation_mode_widget.value == "patch":
        x_start_widget.value -= 1  # Go to the previous patch
    else:
        x_start_widget.value -= nav_config[navigation_mode_widget.value]["increment"]  # Go left by increment

def on_right_button_clicked(button):
    # Adjust the x_start based on the selected navigation mode
    if navigation_mode_widget.value == "patch":
        x_start_widget.value += 1  # Go to the next patch
    else:
        x_start_widget.value += nav_config[navigation_mode_widget.value]["increment"]  # Go right by increment

def save_plot(button):
    fig, ax = update_plot(x_start_widget.value)  # Get the current plot
    save_name = "current_plot.png"  # Define the filename (you can modify this as needed)
    
    fig.savefig(save_name, bbox_inches='tight', pad_inches=0.1, transparent=True)
    print(f"Plot saved as {save_name}")
    
# Create arrow buttons
left_button = widgets.Button(description='◄')
right_button = widgets.Button(description='►')

# Create Save button
save_button = widgets.Button(description='Save Plot')

# Set the save button click event handler
save_button.on_click(save_plot)

# Define widget for the starting value of x-axis (space, time, or patch number)
x_start_widget = widgets.FloatText(value=0.0, description='X start:', continuous_update=False)

# Dropdown to select navigation mode (Space, Time, Patch)
navigation_mode_widget = widgets.Dropdown(
    options=['time', 'space','patch'],
    value='patch',
    description='Nav Mode:',
    disabled=False
)

# Set button click event handlers
left_button.on_click(on_left_button_clicked)
right_button.on_click(on_right_button_clicked)

# Arrange the buttons and widget horizontally
button_box = widgets.HBox([left_button, right_button, navigation_mode_widget, save_button])
ui = widgets.VBox([button_box, x_start_widget])

# Create interactive plot
interactive_plot = widgets.interactive_output(update_plot, {'x_start': x_start_widget, 'navigation_mode': navigation_mode_widget})

# Display the interactive plot and UI
display(ui, interactive_plot)


In [ ]:
trial_summary = plotting.trial_collection(odor_sites, pd.DataFrame(stream_data.lick_onset), cropped_to_length='epoch',
                                            aligned = "choice_cue_time", taken_col='Channel0')

### Plot many sessions 

In [ ]:
from aind_vr_foraging_analysis.utils.parsing import data_access
aggregate_list = []
all_mice_trial_summary = []
# ── Cache path — LOCAL, outside OneDrive ──────────────────────────────────────
results_path = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents\VR foraging\manuscript\results'
aligned = 'succesful_wait_time'  # or 'reward_onset_time', 'odor_onset_time', 'choice_cue_time'
date_string = "2024-9-24"

mouse_list = [
    '754570','754579','754567','754580','754559','754560','754577',
    '754566','754571','754574','754575',
    '754582','745302','745305','745301',
]

for mouse in mouse_list:
    print(f"Mouse {mouse}")

    # This section will look at all the session paths that fulfill the condition
    session_paths = data_access.find_sessions_relative_to_date(
        mouse=mouse,
        date_string=date_string,
        when='on_or_after',
    )
    
    mouse_trial_summary = []

    # Iterate over the session paths and load the data
    # with PdfPages(os.path.join(results_path, f'mouse_{mouse}_lick_rate_plots_{aligned}.pdf')) as pdf:
    for session_path in session_paths:
        with open(os.path.join(session_path, "behavior", "Logs", "tasklogic_input.json")) as f:
            data = json.load(f)
            
        stage = data['stage_name']
        print(stage)
        
        if stage not in ['stageC_v1', 'control', 'data_collection']:
            continue
        try:
            all_epochs, stream_data, data = data_access.load_session(
                session_path, extra=True
            )
        except:
            print(f"Failed to load session {session_path.name}. Skipping.")
            continue
        print(f"Loaded {session_path.name} successfully.")
        
        odor_sites = all_epochs.loc[all_epochs['label'] == 'OdorSite']
        odor_sites['trials_to_leave'] = (
                odor_sites[odor_sites['is_choice'] == 1]
                .groupby('patch_number')['is_choice']
                .transform(lambda x: x.astype(int).cumsum() - x.astype(int).sum() - 1)
            )
        
        odor_sites = odor_sites.groupby('patch_number').filter(lambda x: x['site_number'].nunique() >= 3)

        odor_sites['session'] = session_path.name
        odor_sites['total_sites'] = odor_sites.groupby('patch_number')['site_number'].transform('max') - 1

        total_rewards = odor_sites.is_reward.sum()
        total_licks = len(stream_data.lick_onset)
        metric = total_licks / total_rewards
        
        aggregate_list.append(pd.DataFrame([{
            'mouse': mouse,
            'session': session_path.name,
            'total_rewards': total_rewards,
            'total_licks': total_licks,
            'licks_per_reward': metric
        }]))

        if metric < 20:
            print(f"Warning: Not enough licks per reward in session {session_path.name} (licks/reward = {metric:.2f}). Consider excluding this session from analysis.")
            continue 
        
        max_prob = odor_sites.groupby(['session', 'odor_label'])['reward_probability'].transform('max')
        odor_sites['patch_label'] = max_prob.apply(lambda x: 'odor_90' if x > 0.75 else ('odor_60' if x > 0.3 else 'odor_0'))
        odor_sites.sort_index(inplace=True)
        odor_sites['previous_outcome'] = odor_sites['is_reward'].shift(1)
        odor_sites.loc[odor_sites['site_number'] == 0, 'previous_outcome'] = np.nan
        odor_sites = odor_sites[odor_sites['is_choice'] == 1]
        odor_sites.reset_index(inplace=True)
        lick_rate = lick_rate_calculation(stream_data, 10)
        trial_summary = plotting.trial_collection(odor_sites, lick_rate, cropped_to_length='epoch', window= [-4,5],
                                                    aligned = aligned, taken_col='lick_rate')
        trial_summary['session'] = session_path.name  # keep session label
        mouse_trial_summary.append(trial_summary)

    # ── Plot aggregated across sessions for this mouse ────────────────────
    if len(mouse_trial_summary) == 0:
        print(f"No valid sessions for mouse {mouse}, skipping plot.")
        continue
    
    mouse_df = pd.concat(mouse_trial_summary, ignore_index=True)
    n_sessions = mouse_df['session'].nunique()
    all_mice_trial_summary.append(mouse_df)
    # ── Across-mice grand mean PDF ────────────────────────────────────────────────

    with PdfPages(os.path.join(results_path, f'mouse_{mouse}_lick_rate_plots_{aligned}.pdf')) as pdf:
        # Page 1: aggregated across all sessions
        make_plot(mouse_df, save=pdf, aligned_to=aligned)

        # Page 2+: one page per session (optional, remove if not needed)
        for session_name, session_df in mouse_df.groupby('session'):
            make_plot(session_df, save=pdf, aligned_to=aligned)
            
if all_mice_trial_summary:
    all_mice_df = pd.concat(all_mice_trial_summary, ignore_index=True)
          
if aggregate_list:
    licks = pd.concat(aggregate_list, ignore_index=True)
    licks["session_n"] = licks.groupby("mouse").cumcount() + 1

In [ ]:
with PdfPages(os.path.join(results_path, f'all_mice_lick_rate_plots_{aligned}.pdf')) as pdf:
    make_plot(all_mice_df, aligned_to=aligned, save=pdf)

### Plot many sessions using the lick count

In [ ]:
from aind_vr_foraging_analysis.utils.parsing import data_access
aggregate_list = []
all_mice_trial_summary = []
# ── Cache path — LOCAL, outside OneDrive ──────────────────────────────────────
results_path = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents\VR foraging\manuscript\results'
aligned = 'start_time'  # or 'reward_onset_time', 'odor_onset_time', 'choice_cue_time', "succesful_wait_time"
date_string = "2024-9-24"

mouse_list = [
    '754570','754579','754567','754580','754559','754560','754577',
    '754566','754571','754574','754575',
    '754582','745302','745305','745301',
    # "715866", "713578", "707349", 
    # "716455", "716458","715865","715869","713545","715867","715870"
]

for mouse in mouse_list:
    print(f"Mouse {mouse}")

    # This section will look at all the session paths that fulfill the condition
    session_paths = data_access.find_sessions_relative_to_date(
        mouse=mouse,
        date_string=date_string,
        when='on_or_after',
    )
    
    mouse_trial_summary = []

    # Iterate over the session paths and load the data
    # with PdfPages(os.path.join(results_path, f'mouse_{mouse}_lick_rate_plots_{aligned}.pdf')) as pdf:
    for session_path in session_paths:
        with open(os.path.join(session_path, "behavior", "Logs", "tasklogic_input.json")) as f:
            data = json.load(f)
            
        stage = data['stage_name']
        print(stage)
        
        if stage not in ['stageC_v1', 'control', 'data_collection']:
            continue
        try:
            all_epochs, stream_data, data = data_access.load_session(
                session_path, extra=True
            )
        except:
            print(f"Failed to load session {session_path.name}. Skipping.")
            continue
        print(f"Loaded {session_path.name} successfully.")
        
        odor_sites = all_epochs.loc[all_epochs['label'] == 'OdorSite']
        odor_sites['trials_to_leave'] = (
                odor_sites[odor_sites['is_choice'] == 1]
                .groupby('patch_number')['is_choice']
                .transform(lambda x: x.astype(int).cumsum() - x.astype(int).sum() - 1)
            )
        
        odor_sites = odor_sites.groupby('patch_number').filter(lambda x: x['site_number'].nunique() >= 3)

        odor_sites['session'] = session_path.name
        odor_sites['total_sites'] = odor_sites.groupby('patch_number')['site_number'].transform('max') - 1

        total_rewards = odor_sites.is_reward.sum()
        total_licks = len(stream_data.lick_onset)
        metric = total_licks / total_rewards
        
        aggregate_list.append(pd.DataFrame([{
            'mouse': mouse,
            'session': session_path.name,
            'total_rewards': total_rewards,
            'total_licks': total_licks,
            'licks_per_reward': metric
        }]))

        if metric < 20:
            print(f"Warning: Not enough licks per reward in session {session_path.name} (licks/reward = {metric:.2f}). Consider excluding this session from analysis.")
            continue 
        
        max_prob = odor_sites.groupby(['session', 'odor_label'])['reward_probability'].transform('max')
        odor_sites['patch_label'] = max_prob.apply(lambda x: 'odor_90' if x > 0.75 else ('odor_60' if x > 0.3 else 'odor_0'))
        odor_sites.sort_index(inplace=True)
        odor_sites['previous_outcome'] = odor_sites['is_reward'].shift(1)
        odor_sites.loc[odor_sites['site_number'] == 0, 'previous_outcome'] = np.nan
        odor_sites = odor_sites[odor_sites['is_choice'] == 1]
        odor_sites.reset_index(inplace=True)

        odor_sites['is_reward'] = odor_sites['is_reward'].astype(int)

        # Within each patch, get cumulative reward count
        odor_sites['reward_count_in_patch'] = odor_sites.groupby('patch_number')['is_reward'].cumsum()

        # Count rows since last reward, resetting at patch boundaries
        odor_sites['stops_since_last_reward'] = odor_sites.groupby(['patch_number', 'reward_count_in_patch']).cumcount()

        # Set to NaN for the first reward in each patch (reward_count_in_patch == 1, cumcount == 0)
        # and all rows before the first reward in each patch (reward_count_in_patch == 0)
        odor_sites.loc[odor_sites['site_number'] == 0, 'stops_since_last_reward'] = pd.NA

        odor_sites.reset_index(inplace=True)
        
        trial_summary = lick_functions.collect_lick_trials(
            odor_sites,
            stream_data.lick_onset.index.values,  # adjust if it's a DataFrame column
            aligned=aligned,
        )
        trial_summary['session'] = session_path.name
        trial_summary['measure'] = 'lick'
        mouse_trial_summary.append(trial_summary)
        
        
    # ── Plot aggregated across sessions for this mouse ────────────────────
    if len(mouse_trial_summary) == 0:
        print(f"No valid sessions for mouse {mouse}, skipping plot.")
        continue
    
    mouse_df = pd.concat(mouse_trial_summary, ignore_index=True)
    n_sessions = mouse_df['session'].nunique()
    all_mice_trial_summary.append(mouse_df)
    
    # ── Across-mice grand mean PDF ────────────────────────────────────────────────

    # with PdfPages(os.path.join(results_path, f'mouse_{mouse}_lick_rate_plots_{aligned}.pdf')) as pdf:
    #     # Page 1: aggregated across all sessions
    #     make_plot(mouse_df, save=pdf, aligned_to=aligned)

    #     # Page 2+: one page per session (optional, remove if not needed)
    #     for session_name, session_df in mouse_df.groupby('session'):
    #         make_plot(session_df, save=pdf, aligned_to=aligned)
            
if all_mice_trial_summary:
    all_mice_df = pd.concat(all_mice_trial_summary, ignore_index=True)
    all_mice_df['mouse'] = all_mice_df['session'].str.split('_').str[0]
    all_mice_df['Channel0'] = 1
if aggregate_list:
    licks = pd.concat(aggregate_list, ignore_index=True)
    licks["session_n"] = licks.groupby("mouse").cumcount() + 1

In [ ]:
all_mice_df["datetime"] = pd.to_datetime(
    all_mice_df["session"].str.extract(r"_(\d{8}T\d{6})")[0],
    format="%Y%m%dT%H%M%S"
)

all_mice_df["session_n"] = (
    all_mice_df.groupby("mouse")["datetime"]
    .transform(lambda x: x.rank(method="dense").astype(int))
)

all_mice_df = all_mice_df.drop(columns="datetime")

In [ ]:
import matplotlib.patches as mpatches

# ── Default colors ───────────────────────────────────────
COLOR_0  = color3
COLOR_60 = color2
COLOR_90 = color1

PATCH_COLORS = {
    "odor_0":  COLOR_0,
    "odor_60": COLOR_60,
    "odor_90": COLOR_90,
}
VALID_PATCHES = set(PATCH_COLORS.keys())

def plot_lick_raster(df,
                     patch_col="patch_label",
                     cs_onset=0.0,
                     xlim=(-2, 6),
                     row_height=0.02,
                     bin_size=0.1,
                     order_by=None,
                     show_velocity=False,
                     alignment = 'reward', 
                     color_by=None,        # column to color by, e.g. "patch_label" or "is_reward"
                     palette=None):        # dict mapping values of color_by to colors

    df = df.loc[df['is_choice'] == 1].copy()

    df_licks = df[df["measure"] == "lick"].copy()
    df_vel   = df[df["measure"] == "velocity"].copy()
    df_licks = df_licks[df_licks[patch_col].isin(VALID_PATCHES)].copy()

    # ── Resolve color_by and palette ─────────────────────
    color_by  = color_by or patch_col
    if palette is None:
        if color_by == patch_col:
            palette = PATCH_COLORS
        else:
            # auto-generate colors for unique values
            unique_vals = sorted(df_licks[color_by].unique())
            auto_colors = sns.color_palette("Set2", len(unique_vals))
            palette = {v: auto_colors[i] for i, v in enumerate(unique_vals)}

    # ── Per-trial color lookup ────────────────────────────
    trial_color_col = (
        df_licks[["trial", color_by]]
        .drop_duplicates("trial")
        .set_index("trial")[color_by]
    )

    # ── Trial ordering ────────────────────────────────────
    if order_by:
        trial_meta = (
            df_licks[["trial"] + order_by]
            .drop_duplicates("trial")
            .sort_values(order_by)
        )
        trials_sorted = trial_meta["trial"].values
        divider_keys  = order_by[:-1] if len(order_by) > 1 else order_by
        group_vals    = trial_meta[divider_keys].values
        dividers = [
            i - 0.5
            for i in range(1, len(trials_sorted))
            if not np.array_equal(group_vals[i], group_vals[i - 1])
        ]
    else:
        trials_sorted = np.sort(df_licks['trial'].unique())
        dividers = []

    n_trials   = len(trials_sorted)
    trial_to_y = {t: i for i, t in enumerate(trials_sorted)}

    n_0  = df_licks[df_licks[patch_col] == "odor_0" ]['trial'].nunique()
    n_60 = df_licks[df_licks[patch_col] == "odor_60"]['trial'].nunique()
    n_90 = df_licks[df_licks[patch_col] == "odor_90"]['trial'].nunique()

    # ── Figure layout ─────────────────────────────────────
    fig_height    = max(8, n_trials * row_height + 3.0)
    n_panels      = 3 if show_velocity else 2
    height_ratios = [fig_height - 3, 1.5, 1.5] if show_velocity else [fig_height - 3, 3]

    fig, axes = plt.subplots(
        n_panels, 1,
        figsize=(5, fig_height + (3 if show_velocity else 0)),
        gridspec_kw={"height_ratios": height_ratios, "hspace": 0.08},
        sharex=True
    )
    ax_raster = axes[0]
    ax_rate   = axes[1]
    ax_vel    = axes[2] if show_velocity else None

    bins        = np.arange(xlim[0], xlim[1] + bin_size, bin_size)
    bin_centers = bins[:-1] + bin_size / 2

    # ── Raster ───────────────────────────────────────────
    for trial_id, row_group in df_licks.groupby('trial'):
        y     = trial_to_y[trial_id]
        val   = trial_color_col.get(trial_id)
        color = palette.get(val, "gray")
        lick_times = row_group['times'].values
        ax_raster.vlines(lick_times, y - 0.4, y + 0.4,
                         color=color, linewidth=1.0, alpha=0.85)

    # ── Dividers ──────────────────────────────────────────
    # for d in dividers:
    #     ax_raster.axhline(d, color="black", linewidth=1.0, linestyle=":", alpha=0.5)

    ax_raster.axvline(cs_onset, color="black", linewidth=1.2,
                      linestyle="--", alpha=0.7, zorder=3)
    ax_raster.set_ylim(-0.5, n_trials - 0.5)
    ax_raster.set_ylabel("Trial")

    ytick_idx = np.arange(0, n_trials, 50)
    ax_raster.set_yticks(ytick_idx)
    ax_raster.spines[["top", "right"]].set_visible(False)

    # ── Legend: reflects color_by ─────────────────────────
    legend_handles = [
        mpatches.Patch(color=col, label=str(val))
        for val, col in palette.items()
    ] + [
        plt.Line2D([0], [0], color="black", linestyle="--",
                   linewidth=1.2, label="CS onset"),
    ]
    ax_raster.legend(handles=legend_handles, loc="upper left",
                     framealpha=0.9, fontsize=8, bbox_to_anchor=(1.05, 1),
                     title=color_by)

# ── Lick rate: split by color_by ─────────────────────
    rate_groups = (
        df_licks[["trial", "times", color_by]]
        .copy()
    )

    for val, color in palette.items():
        sub = rate_groups[rate_groups[color_by] == val]
        n_sub = sub["trial"].nunique()
        if n_sub == 0:
            continue
        rate, _ = np.histogram(sub["times"].values, bins=bins)
        rate    = rate / (n_sub * bin_size)
        ax_rate.step(bin_centers, rate, where="mid",
                     color=color, linewidth=1.2, label=str(val))

    ax_rate.axvline(cs_onset, color="black", linewidth=1.2,
                    linestyle="--", alpha=0.7, zorder=3)
    ax_rate.set_ylabel("Lick rate (licks/s)")
    ax_rate.spines[["top", "right"]].set_visible(False)
    ax_rate.yaxis.set_major_locator(plt.MaxNLocator(4, integer=False))
    
    # ── Velocity panel ────────────────────────────────────
    if show_velocity and ax_vel is not None:
        sns.lineplot(data=df_vel, x='times', y='speed', ax=ax_vel,
                     hue=patch_col,
                     palette={k: PATCH_COLORS[k] for k in PATCH_COLORS})
        ax_vel.get_legend().remove()
        ax_vel.axvline(cs_onset, color="black", linewidth=1.2,
                       linestyle="--", alpha=0.7, zorder=3)
        ax_vel.axvspan(0, 2, color="gray", alpha=0.07, zorder=0, linewidth=0)
        ax_vel.set_ylabel("Velocity")
        ax_vel.spines[["top", "right"]].set_visible(False)
        ax_vel.yaxis.set_major_locator(plt.MaxNLocator(4, integer=False))

    axes[-1].set_xlabel(f"Time from {alignment} onset (s)")
    ax_rate.set_xlim(xlim)
    axes[0].set_title(df_licks['session'].iloc[0])

    plt.tight_layout()
    plt.show()
    return fig, axes


In [ ]:

aligned = 'succesful_wait_time'  # or 'odor_onset_time', 'choice_cue_time', "succesful_wait_time"
alignment = 'reward'
all_mice_df = pd.read_csv(os.path.join(results_path, f'all_mice_trial_summary_{aligned}.csv'))

session = '754567_20240930T123236'

session_df = all_mice_df[all_mice_df['session'] == session]
xlim = (-2, 6)

# ── Usage examples ────────────────────────────────────────
fig = plot_lick_raster(session_df,
                 order_by=["odor_sites"],
                 color_by="session",
                 xlim=xlim,
                 alignment = alignment,
                 palette={session: 'black'})
fig[0].savefig(os.path.join(results_path, f'session_{session}_lick_raster_{aligned}_all_black.png'), bbox_inches='tight', pad_inches=0.1, transparent=True)

fig = plot_lick_raster(session_df,
                 order_by=["patch_label", "odor_sites"],
                 color_by="patch_label",
                 xlim=xlim,
                 palette={"odor_0": COLOR_0, "odor_60": COLOR_60, "odor_90": COLOR_90})
fig[0].savefig(os.path.join(results_path, f'session_{session}_lick_raster_{aligned}_by_patch.png'), bbox_inches='tight', pad_inches=0.1, transparent=True)

# Color by reward outcome
fig = plot_lick_raster(session_df,
                 order_by=["is_reward", "site_number"],
                 color_by="is_reward",
                 palette={0: "lightcoral", 1: "darkgreen"},
                 xlim=xlim)
fig[0].savefig(os.path.join(results_path, f'session_{session}_lick_raster_{aligned}_by_reward.png'), bbox_inches='tight', pad_inches=0.1, transparent=True)

# # Color by patch, order by reward then site — auto palette
# plot_lick_raster(session_df,
#                  order_by=["patch_label", "is_reward", "site_number"],
#                  color_by="patch_label",
#                  xlim=(-2, 6))
# fig[0].savefig(os.path.join(results_path, f'session_{session}_lick_raster_{aligned}.png'), bbox_inches='tight', pad_inches=0.1, transparent=True)


In [ ]:
results_path = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents\VR foraging\manuscript\results'

# all_mice_df.to_csv(os.path.join(results_path, f'all_mice_trial_summary_{aligned}.csv'), index=False)

aligned = 'succesful_wait_time'  # or 'reward_onset_time', 'odor_onset_time', 'choice_cue_time', "succesful_wait_time"
all_mice_df = pd.read_csv(os.path.join(results_path, f'all_mice_trial_summary_{aligned}.csv'))

In [ ]:
for mouse in all_mice_df['mouse'].unique():
    with PdfPages(os.path.join(results_path, f'all_mice_lick_rate_plots_{mouse}_rasters.pdf')) as pdf:
        mouse_df = all_mice_df[all_mice_df['mouse'] == mouse]
        for session in mouse_df['session'].unique():
            session_df = mouse_df[mouse_df['session'] == session]
            # Color by patch, order by reward then site — auto palette
            fig = plot_lick_raster(session_df,
                            order_by=["patch_label"],
                            color_by="patch_label")
            pdf.savefig(fig[0])

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'patch_label'
labels   = None
time_interval = 1  # seconds
windows = [
    ("Pre-cue",  -1,  0),
    ("Late",      0, 1),
    ("Consumption",  2,  3),
]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    df['reward_probability'] = df['reward_probability'].round(2)
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) &
            (df['site_number'] == 0) &
            (df['patch_number'] <= 50)
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(3 * len(windows), 4), sharey=True)
set_legend = False
for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    per_mouse['lick_count'] = per_mouse['lick_count'] / time_interval
    if ax == axes[-1]:
        set_legend = True
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette=color_dict_label,
        ax=ax,
        show_legend=set_legend
    )

    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_{aligned}.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'patch_label'
labels   = None
time_interval = 1  # seconds
windows = [
    ("Late",      0, 1),
    ("Consumption",  2,  3),
    ("Pre-cue",  -1,  0),

]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    df['reward_probability'] = df['reward_probability'].round(2)
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 0) &
            (df['site_number'] == 0) &
            (df['patch_number'] <= 50) &
            (df['reward_probability'] > 0.3)
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(3 * len(windows), 4), sharey=True)
set_legend = False
for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    per_mouse['lick_count'] = per_mouse['lick_count'] / time_interval
    if ax == axes[-1]:
        set_legend = True
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette=color_dict_label,
        ax=ax,
        show_legend=set_legend
    )

    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_{aligned}_noreward.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Single mouse, sessions as individual lines + mean
results_one = lick_functions.compute_lick_rate(all_mice_df.loc[(all_mice_df['is_choice'] == 1) & (all_mice_df['is_reward'] == 1) & (all_mice_df['site_number'] == 0) & (all_mice_df['mouse'] == '754570')],
                                 group_var='patch_label',
                                 group_labels=["odor_60", "odor_90"],
                                 t_start=-1, t_end=5, return_sessions=True)
lick_functions.plot_lick_rate(results_one, ["odor_60", "odor_90"], [color2, color1],
            #    plot_type='mean_individual', 
               level='session')

In [ ]:
def apply_filters(df, filters):
    """
    filters: dict where values can be:
        - a scalar         → equality (col == val)
        - a tuple (op, val) → comparison, e.g. ('>', 0.3), ('>=', 0.3), ('<', 5)
    
    Example:
        filters = {
            'is_choice': 1,
            'is_reward': 1,
            'reward_probability': ('>=', 0.3),
            'site_number': ('<', 3),
        }
    """
    ops = {
        '==': lambda col, val: col == val,
        '!=': lambda col, val: col != val,
        '>':  lambda col, val: col >  val,
        '>=': lambda col, val: col >= val,
        '<':  lambda col, val: col <  val,
        '<=': lambda col, val: col <= val,
    }

    mask = pd.Series(True, index=df.index)
    for col, val in filters.items():
        if isinstance(val, tuple):
            op, threshold = val
            mask &= ops[op](df[col], threshold)
        else:
            mask &= (df[col] == val)
    return df.loc[mask]


In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
n_cols = 3
n_rows = 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))

for aligned, ax in zip(['start_time', 'choice_cue_time', 'succesful_wait_time'], axes.flatten()):
    all_mice_df = pd.read_csv(os.path.join(results_path, f'all_mice_trial_summary_{aligned}.csv'))

    cfg = {
            'label':     '',
            'filters':   {
            },
            'group_var':    'Channel0',
            'group_labels': None,
            't_start': -2, 't_end': 6,
        }

    # ── Compute ───────────────────────────────────────────────────────────────────
    all_mice_df['reward_probability'] = all_mice_df['reward_probability'].round(2)

    filtered_df = apply_filters(all_mice_df, cfg['filters'])
    cfg['results'] = lick_functions.compute_lick_rate(
        filtered_df,
        group_var=cfg.get('group_var', None),  # default fallback
        group_labels=cfg['group_labels'],
        window=2,
        t_start=cfg['t_start'],
        t_end=cfg['t_end'],
        return_mice=True,
    )
    # if group_labels was None, store what was inferred
    if cfg['group_labels'] is None:
        cfg['group_labels'] = sorted(filtered_df[cfg['group_var']].dropna().unique())

    lick_functions.plot_lick_rate(
        cfg['results'],
        group_labels=cfg['group_labels'],
        palette=cfg.get('palette', None),
        plot_type='mean_sem',
        level='mouse',
        stats=False,
        ax=ax,
    )
    ax.set_title(cfg['label'])

plt.tight_layout()
plt.savefig(os.path.join(results_path, f'all_mice_lick_count_no_conditions.png'))
plt.show()

In [ ]:
grid_configs = [
    {
        'label':     'Reward — Odor 90 by site',
        'filters':   {
            'is_reward':          1,
            'reward_probability': ('>=', 0.35),
            'patch_label':        'odor_90',
            'total_sites':        ('>=', 3),
            'previous_outcome':   1,  # previous trial was a reward
        },
        'group_var':    'reward_probability',
        'group_labels': None,
        'palette':      'magma_r',
        't_start': 0.05, 't_end': 6,
    },
    {
        'label':     'Reward — Odor 90 by site',
        'filters':   {
            'is_choice':          1,
            'reward_probability': ('>=', 0.35),
            'patch_label':        'odor_90',
            'total_sites':        ('>=', 3),
            'previous_outcome':   1,  # previous trial was a reward
    },
    'group_var':    'reward_probability',
    'group_labels': None,
    'palette':      'magma_r',
    't_start': -1, 't_end': -0.05,
    },
    {
        'label':     'Reward — Odor 90 by site',
        'filters':   {
            'is_choice':          1,
            'is_reward':          0,
            'reward_probability': ('>=', 0.35),
            'patch_label':        'odor_90',
            'total_sites':        ('>=', 3),
            'previous_outcome':   1,  # previous trial was a reward
    },
    'group_var':    'reward_probability',
    'group_labels': None,
    'palette':      'magma_r',
    't_start': -1, 't_end': 2,
    },
    {
        'label':     'No-reward trials by trials to leave',
        'filters':   {
            'is_reward':      0,
            'trials_to_leave': ('>', -5),
            'total_sites':     ('>=', 4),
            'reward_probability': ('==', 0.42),
            # 'previous_outcome':     0,
        },
        'group_var':    'trials_to_leave',
        'group_labels': None,
        'palette':      'Blues',
        't_start': -1, 't_end': 1,
    },
    {
        'label':     'Reward trials by trials to leave',
        'filters':   {
            'is_reward':      1,
            'trials_to_leave': ('>', -5),
            'reward_probability': ('==', 0.42),
            'total_sites':     ('>=', 4),
            # 'previous_outcome':     0,
        },
        'group_var':    'trials_to_leave',
        'group_labels': None,
        'palette':      'Blues',
        't_start': 0.05, 't_end': 5,
    },
    {
        'label':     'Trials (site 0 only)',
        'filters':   {'site_number': 0,
                        'is_reward':       1,
                        'is_choice':       1,
                         'patch_number': ('<', 50)}
        ,
        'group_var': 'patch_label',
        'group_labels': ['odor_60', 'odor_90'],
        'palette':   [color2, color1],
        't_start': -0.1, 't_end': 3,
    },
    {
        'label':     'Trials (site 0 only)',
        'filters':   {'site_number': 0,
                        'is_reward':       1,
                        'is_choice':       1,
                        'patch_number': ('<', 50)},
        'group_var': 'patch_label',
        'group_labels': ['odor_60', 'odor_90'],
        'palette':   [color2, color1],
        't_start': -1, 't_end': 0.05,
    },
    {
        'label':     'Trials (site 0 only)',
        'filters':   {
                        'site_number': 0,
                        'is_reward':       0,
                        'is_choice':       1},
        'group_var': 'patch_label',
        'group_labels': ['odor_60', 'odor_90'],
        'palette':   [color2, color1],
        't_start': -2, 't_end': 2,
    },
    {
        'label':     'Reward trials by previous outcome',
        'filters':   {'patch_label': 'odor_90'},
        'group_var': 'previous_outcome',
        'group_labels': [0, 1],
        'palette':   ['crimson', 'darkgreen'],
        't_start': -1, 't_end': 0.05,
    },
    {
        'label':     'Reward trials by previous outcome',
        'filters':   {'patch_label': 'odor_90',
                        'is_reward': 1},
        'group_var': 'previous_outcome',
        'group_labels': [0, 1],
        'palette':   ['crimson', 'darkgreen'],
        't_start': -0.05, 't_end': 4,
    },
    {
        'label':     'Reward trials by consecutive reward',
        'filters':   {'patch_label': 'odor_90',
                        'is_reward': 1,},
        'group_var': 'consecutive_rewards',
        'group_labels': None,
        'palette':   'RdYlGn',
        't_start': -1, 't_end': -0.05,
    },
    ]


# ── Compute ───────────────────────────────────────────────────────────────────
all_mice_df['reward_probability'] = all_mice_df['reward_probability'].round(2)
for cfg in grid_configs:
    filtered_df = apply_filters(all_mice_df, cfg['filters'])
    cfg['results'] = lick_functions.compute_lick_rate(
        filtered_df,
        group_var=cfg.get('group_var', 'patch_label'),  # default fallback
        group_labels=cfg['group_labels'],
        window=2,
        t_start=cfg['t_start'],
        t_end=cfg['t_end'],
        return_mice=True,
    )
    # if group_labels was None, store what was inferred
    if cfg['group_labels'] is None:
        cfg['group_labels'] = sorted(filtered_df[cfg['group_var']].dropna().unique())

# ── Plot ──────────────────────────────────────────────────────────────────────
n_cols = 3
n_rows = int(np.ceil(len(grid_configs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for ax, cfg in zip(axes, grid_configs):
    lick_functions.plot_lick_rate(
        cfg['results'],
        group_labels=cfg['group_labels'],
        palette=cfg['palette'],
        plot_type='mean_sem',
        level='mouse',
        stats=False,
        ax=ax,
    )
    ax.set_title(cfg['label'])

for ax in axes[len(grid_configs):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(results_path, f'all_mice_lick_count_conditions_{aligned}.svg'), format='svg')
plt.show()

In [ ]:
grid_configs = [

    {
        'label':     'Rewarded, odor_90, fixed to p(R) 0.79',
        'filters':   {
            'is_reward':          1,
            'previous_outcome':   1,  # previous trial was a reward
            'reward_probability': ('<=', 0.42),
            'reward_probability': ('>=', 0.41),
            'patch_number':        ('<', 50),
        },
        'group_var':    'patch_label',
        'group_labels': ['odor_60', 'odor_90'],
        'palette':     [color2, color1],
        't_start': 0.05, 't_end': 5,
    },
    {
        'label':     'Rewarded, odor_90, fixed to p(R) 0.79',
        'filters':   {
            'is_reward':          1,
            'previous_outcome':   1,  # previous trial was a reward
            'reward_probability': ('<=', 0.47),
            'reward_probability': ('>=', 0.46),
            'patch_number':        ('<', 50),
        },
        'group_var':    'patch_label',
        'group_labels': ['odor_60', 'odor_90'],
        'palette':     [color2, color1],
        't_start':0.05, 't_end': 5,
    },
    {
        'label':     'Rewarded, odor_90, fixed to p(R) 0.79',
        'filters':   {
            'is_reward':          1,
            'previous_outcome':   1,  # previous trial was a reward
            'reward_probability': ('<=', 0.42),
            'reward_probability': ('>=', 0.41),
            'patch_number':        ('<', 50),
        },
        'group_var':    'patch_label',
        'group_labels': ['odor_60', 'odor_90'],
        'palette':     [color2, color1],
        't_start': -1, 't_end': -0.05,
    },
    {
        'label':     'Rewarded, odor_90, fixed to p(R) 0.79',
        'filters':   {
            'is_reward':          1,
            'previous_outcome':   1,  # previous trial was a reward
            'reward_probability': ('<=', 0.47),
            'reward_probability': ('>=', 0.46),
            'patch_number':        ('<', 50),
        },
        'group_var':    'patch_label',
        'group_labels': ['odor_60', 'odor_90'],
        'palette':     [color2, color1],
        't_start': -1, 't_end': -0.05,
    }
]
def apply_filters(df, filters):
    """
    filters: dict where values can be:
        - a scalar         → equality (col == val)
        - a tuple (op, val) → comparison, e.g. ('>', 0.3), ('>=', 0.3), ('<', 5)
    
    Example:
        filters = {
            'is_choice': 1,
            'is_reward': 1,
            'reward_probability': ('>=', 0.3),
            'site_number': ('<', 3),
        }
    """
    ops = {
        '==': lambda col, val: col == val,
        '!=': lambda col, val: col != val,
        '>':  lambda col, val: col >  val,
        '>=': lambda col, val: col >= val,
        '<':  lambda col, val: col <  val,
        '<=': lambda col, val: col <= val,
    }

    mask = pd.Series(True, index=df.index)
    for col, val in filters.items():
        if isinstance(val, tuple):
            op, threshold = val
            mask &= ops[op](df[col], threshold)
        else:
            mask &= (df[col] == val)
    return df.loc[mask]

# ── Compute ───────────────────────────────────────────────────────────────────
all_mice_df['reward_probability'] = all_mice_df['reward_probability'].round(2)
for cfg in grid_configs:
    filtered_df = apply_filters(all_mice_df, cfg['filters'])
    cfg['results'] = lick_functions.compute_lick_rate(
        filtered_df,
        group_var=cfg.get('group_var', 'patch_label'),  # default fallback
        group_labels=cfg['group_labels'],
        window=2,
        t_start=cfg['t_start'],
        t_end=cfg['t_end'],
        return_mice=True,
    )
    # if group_labels was None, store what was inferred
    if cfg['group_labels'] is None:
        cfg['group_labels'] = sorted(filtered_df[cfg['group_var']].dropna().unique())

# ── Plot ──────────────────────────────────────────────────────────────────────
n_cols = 3
n_rows = int(np.ceil(len(grid_configs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for ax, cfg in zip(axes, grid_configs):
    lick_functions.plot_lick_rate(
        cfg['results'],
        group_labels=cfg['group_labels'],
        palette=cfg['palette'],
        plot_type='mean_sem',
        level='mouse',
        stats=False,
        ax=ax,
    )
    ax.set_title(cfg['label'])

for ax in axes[len(grid_configs):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(results_path, f'all_mice_lick_count_conditions_{aligned}.svg'), format='svg')
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'patch_label'
labels   = None
time_interval = 1  # seconds
windows = [
    ("Late",      0, 1),
    ("Consumption",  2,  3),
    ("Pre-cue",  -1,  0),

]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    df['reward_probability'] = df['reward_probability'].round(2)
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) &
            (df['previous_outcome'] == 1) &
            ((df['reward_probability'] == 0.46) | (df['reward_probability'] ==0.47))
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(3 * len(windows), 4), sharey=True)
set_legend = False
for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    per_mouse['lick_count'] = per_mouse['lick_count'] / time_interval
    if ax == axes[-1]:
        set_legend = True
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette=color_dict_label,
        ax=ax,
        show_legend=set_legend
    )

    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_{aligned}_reward0.46_0.47.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'patch_label'
labels   = None
time_interval = 1  # seconds
windows = [
    ("Late",      0, 1),
    ("Consumption",  2,  3),
    ("Pre-cue",  -1,  0),

]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    df['reward_probability'] = df['reward_probability'].round(2)
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) &
            (df['previous_outcome'] == 1) &
            ((df['reward_probability'] == 0.46) | (df['reward_probability'] ==0.47))
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(3 * len(windows), 4), sharey=True)
set_legend = False
for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    per_mouse['lick_count'] = per_mouse['lick_count'] / time_interval
    if ax == axes[-1]:
        set_legend = True
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette=color_dict_label,
        ax=ax,
        show_legend=set_legend
    )

    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_{aligned}_reward0.46_0.47.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'patch_label'
labels   = None
time_interval = 1  # seconds
windows = [
    ("Late",      0, 1),
    ("Consumption",  2,  3),
    ("Pre-cue",  -1,  0),
]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    df['reward_probability'] = df['reward_probability'].round(2)
    
    # Get the first lick time per trial (min times where a lick occurred)
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) &
            (df['previous_outcome'] == 1) &
            ((df['reward_probability'] == 0.46) | (df['reward_probability'] == 0.47))
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['times']
        .min()
        .reset_index(name='lick_count')
    )
    
    # Average first lick time across trials within a session
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    
    # Average first lick time across sessions within a mouse
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(3 * len(windows), 4), sharey=False)
set_legend = False
for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    if ax == axes[-1]:
        set_legend = True
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette=color_dict_label,
        ax=ax,
        show_legend=set_legend
    )
    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"First lick timing by {variable}", y=1.02)
plt.tight_layout()
# plt.savefig(os.path.join(results_path, f'box_plots_{variable}_{aligned}_reward0.46_0.47_firstlick.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
variable       = 'patch_label'
bin_width      = 0.1        # seconds per bin
x_max          = 2         # x-axis limit (seconds)
alpha_hist     = 0.6
line_width     = 1.5

filters = dict(
    is_choice           = 1,
    is_reward           = 1,
    previous_outcome    = 1,
)
reward_probs = [0.46, 0.47]

# ── Helper: extract first-lick times per condition label ──────────────────────
def get_first_lick_times(df, label_value):
    df = df.copy()
    df['reward_probability'] = df['reward_probability'].round(2)
    mask = (
        (df['is_choice'] == filters['is_choice']) &
        (df['is_reward'] == filters['is_reward']) &
        (df['previous_outcome'] == filters['previous_outcome']) &
        (df['reward_probability'].isin(reward_probs)) &
        (df[variable] == label_value)
    )
    first_licks = (
        df.loc[mask]
        .groupby(['mouse', 'session', 'odor_sites'])['times']
        .min()
        .dropna()
        .values
    )
    return first_licks


# ── Get all condition labels ───────────────────────────────────────────────────
condition_labels = sorted(all_mice_df[variable].dropna().unique())
bins = np.arange(-x_max, 0 + bin_width, bin_width)
bin_centers = (bins[:-1] + bins[1:]) / 2

# Map each label to a color from your existing palette
colors = [color_dict_label.get(lbl, f'C{i}') for i, lbl in enumerate(condition_labels)]

# ── Per-mouse panels ──────────────────────────────────────────────────────────
mice = sorted(all_mice_df['mouse'].unique())
n_mice = len(mice)
ncols  = min(4, n_mice)
nrows  = int(np.ceil(n_mice / ncols))

fig_mice, axes_mice = plt.subplots(
    nrows, ncols,
    figsize=(3.5 * ncols, 3 * nrows),
    sharey=False, sharex=True
)
axes_mice = np.array(axes_mice).flatten()

for ax, mouse in zip(axes_mice, mice):
    mouse_df = all_mice_df[all_mice_df['mouse'] == mouse]
    for lbl, col in zip(condition_labels, colors):
        fl = get_first_lick_times(mouse_df, lbl)
        if len(fl) == 0:
            continue
        counts, _ = np.histogram(fl, bins=bins)
        density = counts / counts.sum()          # proportion (like 10% y-axis)
        ax.step(bin_centers, density, where='mid',
                color=col, linewidth=line_width, label=str(lbl), alpha=alpha_hist)
        ax.fill_between(bin_centers, density, step='mid', color=col, alpha=0.25)

    ax.set_title(mouse, fontsize=9)
    ax.set_xlim(-x_max, 0)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_xlabel('First-lick time (s)', fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

# hide unused subplots
for ax in axes_mice[n_mice:]:
    ax.set_visible(False)

# shared legend on last visible axis
axes_mice[n_mice - 1].legend(title=variable, fontsize=8, frameon=False)

fig_mice.suptitle(f'First-lick time distribution by {variable} — per mouse', y=1.01)
fig_mice.tight_layout()
fig_mice.savefig(
    os.path.join(results_path, f'firstlick_hist_{variable}_{aligned}_per_mouse.png'),
    bbox_inches='tight', dpi=300
)
plt.show()

# ── Average-of-mice panel ─────────────────────────────────────────────────────
fig_avg, ax_avg = plt.subplots(figsize=(7.5, 4.5))

for lbl, col in zip(condition_labels, colors):
    # collect per-mouse density curves, then average across mice
    density_per_mouse = []
    for mouse in mice:
        mouse_df = all_mice_df[all_mice_df['mouse'] == mouse]
        fl = get_first_lick_times(mouse_df, lbl)
        if len(fl) == 0:
            continue
        counts, _ = np.histogram(fl, bins=bins)
        density_per_mouse.append(counts / counts.sum())

    if not density_per_mouse:
        continue

    density_arr = np.array(density_per_mouse)          # shape: (n_mice, n_bins)
    mean_density = density_arr.mean(axis=0)
    sem_density  = density_arr.std(axis=0) / np.sqrt(len(density_arr))

    ax_avg.step(bin_centers, mean_density, where='mid',
                color=col, linewidth=line_width + 0.5, label=str(lbl))
    ax_avg.fill_between(bin_centers,
                        mean_density - sem_density,
                        mean_density + sem_density, linewidth=0,
                        step='mid', color=col, alpha=0.2)

from scipy.stats import ks_2samp

# ── After collecting density_per_mouse for each label ─────────────────────────
# Collect raw lick times per label for the KS test
# raw_times = {}
# for lbl in condition_labels:
#     all_fl = []
#     for mouse in mice:
#         mouse_df = all_mice_df[all_mice_df['mouse'] == mouse]
#         fl = get_first_lick_times(mouse_df, lbl)
#         all_fl.extend(fl)
#     raw_times[lbl] = np.array(all_fl)

# # Run KS test between each pair of conditions
# from itertools import combinations
# for lbl1, lbl2 in combinations(condition_labels, 2):
#     if len(raw_times[lbl1]) == 0 or len(raw_times[lbl2]) == 0:
#         continue
#     stat, p_value = ks_2samp(raw_times[lbl1], raw_times[lbl2])
#     sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
#     ax_avg.text(
#         0.05, 0.95 - 0.08 * condition_labels.index(lbl1),
#         f'{lbl1} vs {lbl2}: KS={stat:.2f}, p={p_value:.3f} {sig}',
#         transform=ax_avg.transAxes,
#         fontsize=7, verticalalignment='top'
#     )
    
ax_avg.set_xlim(-x_max, 0)
ax_avg.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax_avg.set_xlabel('First-lick time (s)')
ax_avg.set_ylabel('Proportion of trials')
ax_avg.legend(title=variable, frameon=False,bbox_to_anchor=(1.05, 1.05))
ax_avg.spines[['top', 'right']].set_visible(False)

fig_avg.tight_layout()
fig_avg.savefig(
    os.path.join(results_path, f'firstlick_hist_{variable}_{aligned}_avg_mice.png'),
    bbox_inches='tight', dpi=300
)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'site_number'
labels   = None

windows = [

    # ("Post-cue",  0.0,  0.5),
    ("Pre-cue",  -1, 0),
    ("Late",      1,  2),
]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) & 
            (df['previous_outcome'] == 1) &
            (df['reward_probability'] == 0.54) &
            (df['site_number'] >= 4) &
            (df['site_number'] <= 9) &
            # (df['reward_probability']< 0.9) &
            (df['patch_label']  == 'odor_90')
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(10,5), sharey=True)

for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette='magma_r',
        ax=ax,
    )
    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_reward_trials_odor_90_{aligned}.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'trials_to_leave'
labels   = None
time_interval = 2
windows = [

    # ("Post-cue",  0.0,  0.5),
    ("Pre-cue",  -1, 1),
    ("Late",      2,  3),
]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) & 
            (df['previous_outcome'] == 0) &
            (df['reward_probability'] >= 0.3) &
            (df['trials_to_leave'] >= -4) &
            # (df['reward_probability']< 0.9) &
            (df['patch_label']  == 'odor_90')
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(10,5), sharey=False)

for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    per_mouse['lick_count'] = per_mouse['lick_count'] / time_interval
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette='Blues',
        ax=ax,
    )
    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_reward_trials_odor_90_{aligned}_reward.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'reward_probability'
labels   = None

windows = [

    # ("Post-cue",  0.0,  0.5),
    ("Pre-cue",  -1, 0),
    ("Late",      2,  3),
]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) & 
            (df['previous_outcome'] == 1) &
            (df['reward_probability'] >= 0.28) &
            # (df['reward_probability']< 0.9) &
            (df['patch_label']  == 'odor_90')
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(10,5), sharey=False)

for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette='magma_r',
        ax=ax,
    )
    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_reward_trials_odor_90_{aligned}.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'reward_probability'
labels   = None
time_interval = 2  # seconds
windows = [

    # ("Post-cue",  0.0,  0.5),
    ("Pre-cue",  -1, 0),
    ("Late",      0,  2),
]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) & 
            (df['previous_outcome'] == 1) &
            (df['reward_probability'] >= 0.28) &
            # (df['reward_probability']< 0.9) &
            (df['patch_label']  == 'odor_90')
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(10,5), sharey=False)

for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    per_mouse['lick_count'] = per_mouse['lick_count'] / time_interval
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette='magma_r',
        ax=ax,
    )
    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_reward_trials_odor_90_{aligned}_no_reward.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
all_mice_df['consecutive_rewards'] = np.where((all_mice_df['consecutive_rewards'] == 0)&(all_mice_df['site_number'] != 0), -all_mice_df['consecutive_failures'], all_mice_df['consecutive_rewards'])

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
variable = 'consecutive_rewards'
labels   = None

windows = [

    # ("Post-cue",  0.0,  0.5),
    ("Pre-cue",  -1, 0),
    ("Late",      2,  3),
]

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) & 
            (df[variable] <= 3) &
            (df[variable] >= -3) &
            (df[variable] != 0) &
            (df['reward_probability'] >= 0.3) &
            # (df['reward_probability']< 0.9) &
            (df['patch_label']  == 'odor_90')
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(windows), figsize=(10,5), sharey=False)

for ax, (name, t0, t1) in zip(axes, windows):
    per_mouse, per_session = aggregate_window(all_mice_df, t0, t1, variable)
    lick_functions.plot_lick_count_by_condition(
        per_mouse,
        group_var=variable,
        group_labels=labels,
        condition='mouse',
        palette='RdYlGn',
        ax=ax,
    )
    ax.set_title(f"{name}\n[{t0}, {t1}]")

fig.suptitle(f"Lick counts by {variable}", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_path, f'box_plots_{variable}_reward_trials_odor_90_{aligned}.png'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
from scipy import stats

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) & 
            (df['previous_outcome'] == 1) &
            (df['reward_probability'] >= 0.3) &
            (df['patch_label']  == 'odor_90')
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
variable = 'reward_probability'
per_mouse, per_session = aggregate_window(all_mice_df, 0.-0.5, 0.5, variable)
        
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for i, (mouse, grp) in enumerate(per_mouse.groupby('mouse')):
    color = colors[i % len(colors)]
    
    x = grp['reward_probability']
    y = grp['lick_count']
    
    slope, intercept, r, p, se = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = intercept + slope * x_line
    
    ax.scatter(x, y, color=color, s=30, alpha=0.6, zorder=5)
    ax.plot(x_line, y_line, color=color, linewidth=1.5,
            label=f'{mouse}  r={r:.2f}, p={p:.3f}')

ax.set_xlabel('Reward probability')
ax.set_ylabel('Lick count')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., ncol=2)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
from scipy import stats

# ── Aggregation ───────────────────────────────────────────────────────────────
def aggregate_window(df, window_start, window_end, variable):
    per_trial = (
        df.loc[
            df['times'].between(window_start, window_end) &
            (df['is_choice'] == 1) &
            (df['is_reward'] == 1) 
            &  (df['reward_probability'].isin([0.54]))
            &  (df['consecutive_rewards']  != 0)
        ]
        .groupby(['mouse', 'session', 'odor_sites', variable])['Channel0']
        .sum()
        .reset_index(name='lick_count')
    )
    per_session = (
        per_trial.groupby(['mouse', 'session', variable])['lick_count']
        .mean()
        .reset_index()
    )
    per_mouse = (
        per_session.groupby(['mouse', variable])['lick_count']
        .mean()
        .reset_index()
    )
    return per_mouse, per_session

# ── Plot ──────────────────────────────────────────────────────────────────────
variable = 'consecutive_rewards'
per_mouse, per_session = aggregate_window(all_mice_df, 0.-0.5, 0.5, variable)
        
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for i, (mouse, grp) in enumerate(per_mouse.groupby('mouse')):
    color = colors[i % len(colors)]
    
    x = grp[variable]
    y = grp['lick_count']
    
    slope, intercept, r, p, se = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = intercept + slope * x_line
    
    ax.scatter(x, y, color=color, s=30, alpha=0.6, zorder=5)
    ax.plot(x_line, y_line, color=color, linewidth=1.5,
            label=f'{mouse}  r={r:.2f}, p={p:.3f}')

ax.set_xlabel('Previous outcome')
ax.set_ylabel('Lick count')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
sns.despine()
plt.tight_layout()
plt.show()